In [ ]:
import pickle # Load refs and annotations
import json
import os
import pandas as pd
import numpy as np
import pprint
import json
import cv2
import random

from typing import Any, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.utils.tensorboard import SummaryWriter

import torchvision
import torchvision.transforms as transforms
from torchvision.utils import draw_bounding_boxes
from torchvision import models
import torchmetrics

import pytorch_lightning as pl
from pytorch_lightning.utilities.types import STEP_OUTPUT

from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import CLIPProcessor, CLIPModel

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import clip
from ultralytics import YOLO
from PIL import Image, ImageDraw

from ipywidgets import FloatProgress
import math 
from torch.nn.modules.batchnorm import _BatchNorm

In [ ]:
device = torch.device('cuda')

In [ ]:
clip_model, clip_preprocess = clip.load("RN50", device=device)

In [ ]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_area = max(0, x2 - x1 + 1) * max(0, y2 - y1 + 1) # +1 to avoid max(0,0) therefore avoiding 

    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    union_area = box1_area + box2_area - intersection_area

    return intersection_area / union_area

In [ ]:
class MetricMeter:
    def __init__(self, name="Default", threshold = 0.5, log_dir='./logs/default_run'):
        self.name = name
        self.metrics = []
        self.epochs = []
        self.threshold = threshold
        self.reset()
        self.best_cases_iou = []  # To store the best cases
        self.worst_cases_iou = []
        self.best_cases_sim = []  # To store the best cases
        self.worst_cases_sim = []
        self.writer = SummaryWriter(log_dir=log_dir)

# TODO add SummaryWriter to plot

    def reset(self):
        self.count = 0
        self.iou = 0
        self.epoch = 0
        self.correct_bboxes, self.overall = 0,0
        self.semantic = 0
        self.metrics = []
    
    def add_best_iou(self, item):
        if len(self.best_cases_iou) < 5:
            self.best_cases_iou.append(item)
        else:
            min_best_case = min(self.best_cases_iou, key=lambda x: (x['iou']))
            if item['iou'] > min_best_case['iou']:
                self.best_cases_iou.remove(min_best_case)
                self.best_cases_iou.append(item)
    
    def add_worst_iou(self, item):
        if len(self.worst_cases_iou) < 5:
            self.worst_cases_iou.append(item)
        else:
            max_worst_case = max(self.worst_cases_iou, key=lambda x: (x['iou']))
            if (item['iou'] < max_worst_case['iou']):
                self.worst_cases_iou.remove(max_worst_case)
                self.worst_cases_iou.append(item)
    
    def add_best_confidence(self, item):
        if len(self.best_cases_sim) < 5:
            self.best_cases_sim.append(item)
        else:
            min_best_case = min(self.best_cases_sim, key=lambda x: (x['confidence']))
            if item['confidence'] > min_best_case['confidence']:
                self.best_cases_sim.remove(min_best_case)
                self.best_cases_sim.append(item)
    
    def add_worst_confidence(self, item):
        if len(self.worst_cases_sim) < 5:
            self.worst_cases_sim.append(item)
        else:
            max_worst_case = max(self.worst_cases_sim, key=lambda x: (x['confidence']))
            if (item['confidence'] < max_worst_case['confidence']):
                self.worst_cases_sim.remove(max_worst_case)
                self.worst_cases_sim.append(item)


    def update(self, iou, confidence,filename,bbox_e,bbox_gt):
        self.count += 1
        self.iou += iou
        if(iou >= self.threshold):
            self.correct_bboxes += 1
        self.semantic += confidence

        iteration = {
            'run_loc_acc': self.iou / self.count,
            'run_gro_acc': self.correct_bboxes / self.count,
            'run_sem_acc': self.semantic / self.count,
        }

        item = {
            'iou': iou,
            'confidence':confidence,
            'ground_truth': bbox_gt,
            'candidate': bbox_e,
            'path':filename
        }
        self.metrics.append([iteration])
        self.add_best_confidence(item)
        self.add_worst_confidence(item)
        self.add_best_iou(item)
        self.add_worst_iou(item)
        self.writer.add_scalar('Localization Loss', self.iou / self.count, self.count)
        self.writer.add_scalar('Grounding Accuracy',  self.correct_bboxes / self.count, self.count)
        self.writer.add_scalar('Semantic Similarity', self.semantic / self.count, self.count)

    def new_epoch(self):
        self.epoch += 1
        self.writer.add_scalar('Localization Loss', self.iou / self.count, self.epoch)
        self.writer.add_scalar('Grounding Accuracy',  self.correct_bboxes / self.count, self.epoch)
        self.writer.add_scalar('Semantic Similarity', self.semantic / self.count, self.epoch)
        self.epochs.append(self.metrics,self.best_cases_iou,self.best_cases_sim,self.worst_cases_iou,self.worst_cases_sim)
        self.reset()

    def __repr__(self):
        text = f"{self.name}: {self.avg:.8f}"
        return text
    
    def print_iteration(self):
        print(f"Localization accuracy = {self.iou / self.count}, Grounding Accuracy = {self.correct_bboxes / self.count}, Semantic Similarity = {self.semantic / self.count}")

    def get_best_iou_cases(self):
        return sorted(self.best_cases_iou, key=lambda x: x['iou'], reverse=True)

    def get_worst_iou_cases(self):
        return sorted(self.worst_cases_iou, key=lambda x: x['iou'])

    def get_best_pred_cases(self):
        return sorted(self.best_cases_sim, key=lambda x: x['confidence'], reverse=True)

    def get_worst_pred_cases(self):
        return sorted(self.worst_cases_sim, key=lambda x: x['confidence'])
    
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group["lr"]

In [ ]:
with open("./refcocog/annotations/refs(umd).p", "rb") as fp:
  refs = pickle.load(fp)

# 'annotations' will be a dict object mapping the 'annotation_id' to the 'bbox' to make search faster
with open("./refcocog/annotations/instances.json", "rb") as fp:
  data = json.load(fp)
  annotations = dict(sorted({ann["id"]: ann["bbox"] for ann in data["annotations"]}.items()))

In [ ]:
def getcaption(elem):
    li = []
    for e in elem["sentences"]:
        li.append(e['raw'])
    return li

In [ ]:
class RefCOCOG_noproc(Dataset):
    """
    Args:
        The dataset will be the raw data wothput any tipe of preprocessing
        {
            'file_name': 
            'caption':
            'ann_id': needed to extract the relative bbox from the .json file
            'bbox': values are set like following:
                - x 
                - y
                - width 
                - height
        }
    """
    def __init__(self, refs, model, preprocess, annotations, split="train", device = 'cuda', count = 160):
        
        self.clip_model, self.clip_preprocess = model, preprocess
        self.device = device
        #self.images = []
        self.texts = []
        self.filepaths =[]
        self.gt = []
        self.cls = []
        temp = 0
        for elem in [d for d in refs if d["split"]==split]:
            
            # Retrieve Single image
            file_name = os.path.join("./refcocog/images/", f'{"_".join(elem["file_name"].split("_")[:3])}.jpg')
            #image = Image.open(file_name)

            # Retrieve possible ground truth measures
            cls = elem['category_id']
            gt = annotations[elem['ann_id']]
            bbox_tnsor = torch.tensor(gt, device=self.device)
            new_bbox = box_convert(bbox_tnsor, 'xywh', 'xyxy')
            # Get all texts related to the picture
            sentences = elem['sentences']
            # for i in sentences:
            self.texts.append(clip.tokenize(sentences[0]['raw']))
            #self.images.append(self.clip_preprocess(image))
            self.gt.append(new_bbox)
            self.cls.append(cls)
            self.filepaths.append(file_name)
            # temp += 1
            # if (temp > count):
            #     break

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        #images = self.images[idx]
        gt = self.gt[idx]
        cls = self.cls[idx]
        filename = self.filepaths[idx]
        return text, gt, cls, filename#, images

    def __call__(self, idx):
        print(json.dumps(self.dataset[idx], indent=4))


In [ ]:
# create dataset and dataloader
print("----------------------Processing train split----------------------------")
dataset_train = RefCOCOG_noproc(refs, clip_model, clip_preprocess, annotations, split="train")
dataloader_train = DataLoader(dataset_train, batch_size=16)
len_train = len(dataset_train)
print(f"Numero esempi in train = {len_train}")

print("----------------------Processing test split-----------------------------")
dataset_test = RefCOCOG_noproc(refs, clip_model, clip_preprocess, annotations, split="test")
dataloader_test = DataLoader(dataset_test, batch_size=16)
len_test = len(dataset_test)
print(f"Numero esempi in train = {len_test}")

print("----------------------Processing eval split-----------------------------")
dataset_eval = RefCOCOG_noproc(refs, clip_model, clip_preprocess, annotations, split="val")
dataloader_eval = DataLoader(dataset_eval, batch_size=16)
len_eval = len(dataset_eval)
print(f"Numero esempi in eval = {len_eval}")

print("------------------------------------------------------------------------")


## Eval

In [ ]:
##########################################################
# YolottoClip - Just for Zero-Shotting the dataset
##########################################################

class BaselineYOLO():
    def __init__(self, clip_model, clip_preprocess, device = "cpu"):
        self.yolo = YOLO("yolov8n.pt")
        self.device = device
        self.clip_model, self.clip_preprocess = clip_model, clip_preprocess
        self.processing = []
        self.preprocessed = []
        self.similarities = []

    def infer_bboxes(self, image_path):
        results = self.yolo(image_path, verbose=False)
        bboxes = results[0].boxes.xyxy
        # cls = results[0].boxes.cls
        return bboxes#, cls

    def encode_image(self, image):
        with torch.no_grad():
            image_features = self.clip_model.encode_image(image)
        return image_features
    
    def preprocess_images(self, cropped_images):
        # preprocess with CLIP each cropped PIL image(converts each image in a image
        # of size [3,224,224])
        processing = []
        for image in cropped_images:
            processed_img = self.clip_preprocess(image).to(self.device)
            processing.append(processed_img)
        preprocessed = torch.stack(self.processing).to(self.device) # return a single tensor
        return preprocessed

    def encode_text(self, text):
        # Encode the text using the CLIP model
        with torch.no_grad():
            text_features = self.clip_model.encode_text(text)
        return text_features
    
    def calculate_best_bbox(self, image_path, caption):
        self.similarities = []
        best_score = 0
        best_bbox = None
        images = []
        ex_bbox = []
        clss = []

        ex_bbox,clss = self.infer_bboxes(image_path)

        for bbox in ex_bbox:
            temp = cv2.imread(image_path)
            image = np.zeros((temp.shape[0], temp.shape[1], temp.shape[2]), dtype=np.uint8)
            image[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])] = temp[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])]
            image = Image.fromarray(image)
            images.append(image)
            #image = self.clip_preprocess(image).unsqueeze(0).to(device)

        self.preprocess_images(images)

        with torch.no_grad():
            image_features = self.encode_image(self.preprocessed)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_features = self.encode_text(caption).float()
            text_features /= text_features.norm(dim=-1, keepdim=True)
            logits_per_image = image_features @ text_features.t()
            matching_score = logits_per_image.cpu().numpy()[0]

            self.similarities.append(matching_score)

        print(len(self.similarities))
        best_bbox = np.argmax(self.similarities)
        best_logit = self.similarities[best_bbox]
        print(best_bbox)

        xmin = ex_bbox[best_bbox,0]
        ymin = ex_bbox[best_bbox,1]
        xmax = ex_bbox[best_bbox,2]
        ymax = ex_bbox[best_bbox,3]
        pred_bbox = torch.tensor([xmin, ymin, xmax, ymax], device=self.device)

        if matching_score > best_score:
            best_score = matching_score
            best_bbox = bbox

        return best_logit, pred_bbox
    